In [1]:
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

In [2]:
data = pd.read_excel("../data.xlsx", index_col=0)
data.columns = [col.split(' (')[0] if ":" in col else col for col in data.columns]
data['Normal fertilization'] = data['Normal fertilization'].replace({0: 'abnormal', 1: 'normal'})
data['Normal fertilization'] = pd.Categorical(data['Normal fertilization'], ['abnormal', 'normal'])
data

,Normal fertilization,Patient age,Partner age,Previous IVF cycles,Infertility type,Male factors cause infertility,Female factors cause infertility,Years of infertility,Body Mass Index (BMI),Follicle Stimulating Hormone (FSH),...,IM (Immotile) sperm count before semen optimization treatment,Total sperm count before semen optimization treatment (x10^6),Volume after semen optimization treatment,Concentration after semen optimization treatment,PR sperm count after semen optimization treatment,NP sperm count after semen optimization treatment,IM sperm count after semen optimization treatment,Number of oocyte for IVF,Number of oocyte for ICSI,Number of MII oocyte for ICSI
ID,,,,,,,,,,,,,,,,,,,,,
1,normal,26,40,1.0,1,0,1,0.5,25.3,1.91,...,47,120.40,0.6,4,86,5,9,6,0,0
2,normal,29,32,1.0,0,1,0,3.0,19.9,7.60,...,60,22.10,0.5,2,85,5,10,0,4,4
3,normal,26,29,1.0,0,1,0,3.0,17.1,6.98,...,69,24.00,0.5,2,80,10,10,0,4,4
4,normal,31,31,1.0,1,1,1,5.0,20.4,4.66,...,45,85.80,0.5,3,90,5,5,7,0,0
5,normal,26,28,1.0,0,1,0,2.0,22.9,7.21,...,83,59.74,0.5,2,70,10,20,0,9,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6159,normal,32,32,2.0,1,0,1,1.0,20.0,15.24,...,49,159.30,0.5,2,90,5,5,3,0,0
6160,normal,29,35,1.0,0,0,1,3.0,18.9,5.20,...,46,407.40,0.5,3,90,5,5,7,0,0
6161,normal,29,33,1.0,0,1,1,2.0,21.6,6.08,...,58,24.30,0.5,2,85,5,10,5,0,0


In [3]:
discrete_value_columns = data.columns[data.nunique() < 10].drop("Normal fertilization")
consecutive_value_columns = data.columns[data.nunique() >= 10]

In [4]:
df = pd.DataFrame()
for column in discrete_value_columns:
    _data = data.copy()

    _df = pd.crosstab(_data[column], _data['Normal fertilization'], rownames=['value']).reset_index()

    stat, p, dof, expected =  chi2_contingency(_df[_df.columns.drop('value')])
    _df['stat'], _df['P'] = stat, p

    for efficacy in _data['Normal fertilization'].cat.categories:
        _proportion = (round(_df[efficacy] / _df[efficacy].sum() * 100, 2)).astype(str)
        _df[efficacy] = _df[efficacy].astype(str) + '(' + _proportion + '%)'

    _df['feature'] = column
    _df.set_index(['feature', 'value'], inplace=True)
    df = pd.concat([df, _df])
df.to_csv('discrete_feature_distribution.csv')
df


Normal fertilization                                        abnormal  \
feature                                           value                
Infertility type                                  0.0    125(42.81%)   
                                                  1.0    167(57.19%)   
Male factors cause infertility                    0.0    156(53.42%)   
                                                  1.0    136(46.58%)   
Female factors cause infertility                  0.0     34(11.64%)   
                                                  1.0    258(88.36%)   
Classification of T                               0.0      33(14.6%)   
                                                  1.0     193(85.4%)   
Classification of P                               0.0    101(37.41%)   
                                                  1.0    169(62.59%)   
Type of semen                                     1.0    283(96.92%)   
                                                  2.0       9(3.08%)   
Liquefying time                                   0.0     258(95.2%)   
                                                  1.0       13(4.8%)   
Sperm source                                      1.0    274(93.84%)   
                                                  2.0      18(6.16%)   
Concentration before semen optimization treatment 1.0      24(8.22%)   
                                                  2.0       1(0.34%)   
                                                  3.0       4(1.37%)   
                                                  4.0       8(2.74%)   
                                                  5.0    255(87.33%)   
Concentration after semen optimization treatment  1.0      24(8.22%)   
                                                  2.0    148(50.68%)   
                                                  3.0     86(29.45%)   
                                                  4.0      10(3.42%)   
                                                  5.0      24(8.22%)   

Normal fertilization                                           normal  \
feature                                           value                 
Infertility type                                  0.0    2652(45.17%)   
                                                  1.0    3219(54.83%)   
Male factors cause infertility                    0.0    3111(52.99%)   
                                                  1.0    2760(47.01%)   
Female factors cause infertility                  0.0    1320(22.48%)   
                                                  1.0    4551(77.52%)   
Classification of T                               0.0      505(9.67%)   
                                                  1.0    4718(90.33%)   
Classification of P                               0.0    1898(34.59%)   
                                                  1.0    3589(65.41%)   
Type of semen                                     1.0    5765(98.19%)   
                                                  2.0      106(1.81%)   
Liquefying time                                   0.0     5202(95.0%)   
                                                  1.0       274(5.0%)   
Sperm source                                      1.0    5490(93.51%)   
                                                  2.0      381(6.49%)   
Concentration before semen optimization treatment 1.0      509(8.67%)   
                                                  2.0       26(0.44%)   
                                                  3.0        88(1.5%)   
                                                  4.0      104(1.77%)   
                                                  5.0    5144(87.62%)   
Concentration after semen optimization treatment  1.0      510(8.69%)   
                                                  2.0    2751(46.86%)   
                                                  3.0    2200(37.47%)   
                                                  4.0      124(2.11%)   
            

In [5]:
df = pd.DataFrame()
for column in consecutive_value_columns:
    _data = data[data[column].notna()]

    _df = {}
    column_values = []
    for efficacy in _data['Normal fertilization'].cat.categories:
        _column_values = _data.loc[_data['Normal fertilization'] == efficacy, column]
        _df[efficacy] = '{}({}-{})'.format(round(_column_values.median(), 2), _column_values.min(), _column_values.max())
        column_values.append(_column_values)
    _df = pd.DataFrame([_df])

    stat, p = mannwhitneyu(*column_values)
    _df['stat'], _df['P'] = stat, p
    _df['feature'] = column
    _df.set_index('feature', inplace=True)
    df = pd.concat([df, _df])
df.to_csv('consecutive_feature_distribution.csv')
df

,abnormal,normal,stat,P
feature,,,,
Patient age,37.0(24-49),33.0(20-50),1152288.0,2.273627e-23
Partner age,37.0(26-69),34.0(22-69),1063238.5,3.545042e-12
Previous IVF cycles,2.0(1.0-10.0),1.0(1.0-11.0),1189343.5,2.353571e-44
Years of infertility,3.0(0.4-19.0),3.0(0.1-19.0),912005.0,6.074173e-02
Body Mass Index (BMI),21.4(16.0-34.5),21.5(2.4-52.6),852815.5,9.339993e-01
Follicle Stimulating Hormone (FSH),8.18(1.28-51.11),6.61(0.1-45.14),1145787.0,1.223372e-22
Luteinizing hormone (LH),4.79(0.5-34.87),5.15(0.07-87.1),783422.5,1.295372e-02
Prolactin (PRL),15.9(2.57-95.04),15.11(0.28-156.55),610901.5,8.669013e-01
Estradiol (E2),42.0(5.0-504.0),42.0(3.36-4768.0),856944.5,9.940578e-01
